# comma2k19 → false positives per hour

**What this notebook does:** downloads comma2k19 highway footage, remuxes it to playable video at
a *derived* frame rate, and scores it with BADAS-Open on a T4 to produce per-frame traces.

**What it does NOT do:** it produces **no headline number**. Traces come back to the Mac and
`eval/fp_rate.py` — the committed code that already reproduces Nexar's 92.3 FP/hour — computes the
rate there. One code path, one set of numbers.

**Runtime: GPU (T4).** Unlike `colab_dada_extract.ipynb`, this notebook runs the model.

**Why it exists.** README §31 makes FP/hour the metric that decides the product, target
< 0.1/hour. We measure 92.3 — but on **0.899 h** of Nexar negatives, where the smallest
expressible non-zero rate is **1.1/hour**. The target is *arithmetically unmeasurable* on that
corpus. comma2k19 is 33 h and is NEW_PLAN §8.1's only genuinely-independent evidence tier.

**Licence:** comma2k19 is **MIT**. Nexar clips are used here only to verify hardware equivalence.

---

### Order of cells is mandatory and enforced in code

`§1 identity gates` → `§2 equivalence gate` → `§3 acquire` → `§4 frame-rate gate` → `§5 pilot`
→ `§6 full run` → `§7 return`.

**§2 writes a sentinel. §5 and §6 refuse to run without it.** This is not a convention you are
asked to respect — scoring cells assert on it and raise. The reason: the 0.9733 alert threshold
was derived from scores computed on Apple MPS. If a CUDA T4 scores differently, every comma2k19
number is measured against a threshold that means something else, comparability to 92.3 is gone,
and **nothing would raise an error**.


## 1 — Environment, and three identity gates

Cheap checks first, so a surprise surfaces before anything expensive runs.

- **GATE A** the checkpoint is byte-identical to the one every committed number used
- **GATE B** the vendored scoring code is byte-identical to the repo's
- the library stack is **recorded**, not forced

🔴 **The checkpoint source in `progress.md` §21.11 is wrong twice** (verified 2026-09-19):
it names `getnexar/BADAS-Open`, which is the **GitHub** org — on Hugging Face the model is
`nexar-ai/BADAS-Open` — and it calls that mirror *ungated* when it is **gated**. This repo's own
`.gitignore` said "gated HF download" and was right. HF returns `401` for both a missing repo and
a gated one, so the two mistakes hid each other.

The licence is Apache 2.0, free for research **and** commercial use, so only *access* is the
problem. §1 therefore takes the checkpoint from **Drive** if it is there, and falls back to HF
with a token. Either way **GATE A's digest decides** — the source is interchangeable, the bytes
are not.

**Why the stack is recorded rather than pinned.** An earlier version of this cell halted unless
`torch`/`transformers`/`numpy` matched the Mac exactly. Running it showed that was wrong:

1. Colab **preloads** these modules, so `pip install` cannot change the running kernel — the
   check becomes a restart loop, not a gate.
2. Replacing Colab's torch is a ~2.5 GB download that may not match the CUDA 12.8 driver, and
   the Mac's versions may have no wheels for Colab's Python 3.13.
3. **It answers the wrong question.** §2 asks *"is a number produced here comparable to the
   committed 92.3?"* If it passes on Colab's own stack, comparability is demonstrated
   empirically — and that is the stack the comma2k19 numbers will actually be produced on.

The cause of a mismatch matters only for **diagnosis**, and §2's pixel fingerprint supplies that
for the price of one clip.


In [ ]:
# === §1 — environment + identity gates. Cheap, and they HALT. ===
import hashlib, json, os, shutil, subprocess, sys, tarfile, time

CKPT_SHA = "6b1ba91504542582412fee5100a17d6e06c87cb09619efec2efc34484f7042aa"
CKPT_BYTES = 3_979_436_545
PINS = {"torch": "2.14.0", "transformers": "5.17.0", "numpy": "2.4.6"}   # the Mac's stack

REPO = "/content/repo"
DRIVE_DIR = "/content/drive/MyDrive/crash_detection_colab"   # bundle lives here
NEXAR_CANDIDATES = [
    "/content/drive/MyDrive/nexar/test-public",
    "/content/drive/MyDrive/test-public",
    "/content/drive/MyDrive/nexar_test_public",
]

def sha256(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(block), b""):
            h.update(blk)
    return h.hexdigest()

from google.colab import drive
drive.mount("/content/drive")

# ---- stage the repo. NOT git clone: 10 commits are unpushed, so origin/main lacks
# ---- score_external.py and fp_rate.py entirely. The bundle carries the real code.
bundle = os.path.join(DRIVE_DIR, "colab_bundle.tar.gz")
if not os.path.exists(bundle):
    raise SystemExit(
        f"No bundle at {bundle}. On the Mac run:\n"
        "  ~/envs/badas/bin/python scripts/make_colab_bundle.py\n"
        "then upload runs/colab_bundle.tar.gz to that Drive folder. "
        "Do not git clone -- origin/main is 10 commits stale and lacks the scoring code.")

os.makedirs(REPO, exist_ok=True)
with tarfile.open(bundle) as tar:
    tar.extractall(REPO)
print(f"staged bundle -> {REPO}")

# ---- GATE B: the vendored code is what left the Mac -------------------------------------
with open(os.path.join(REPO, "BUNDLE_MANIFEST.json")) as f:
    man = json.load(f)
bad = []
for rel, rec in sorted(man["clips"].items()):
    p = os.path.join(REPO, rel)
    if not os.path.exists(p):
        bad.append((rel, "MISSING"))
    elif os.path.getsize(p) != rec["bytes"]:
        bad.append((rel, f"SIZE {os.path.getsize(p)} != {rec['bytes']}"))
    elif sha256(p) != rec["sha256"]:
        bad.append((rel, "SHA-256 MISMATCH"))
if bad:
    for rel, why in bad:
        print(f"  {rel:<50} {why}")
    raise SystemExit("GATE B FAILED -- the staged code is not the repo's. Re-upload the bundle; "
                     "do not improvise around this.")
print(f"GATE B  PASS -- {man['n']} files byte-identical to the repo")

# ---- Record the stack. DO NOT force it to match the Mac's. -------------------------------
# An earlier version of this cell HALTed unless torch/transformers/numpy matched the Mac
# exactly. That was wrong, for three reasons found by actually running it:
#   1. Colab preloads these modules, so pip cannot change the RUNNING kernel -- only a restart
#      can, which makes the check a restart loop rather than a gate.
#   2. Replacing Colab's torch means a ~2.5 GB download that may not match the CUDA 12.8
#      driver, and the Mac's exact versions may have no wheels for Colab's Python 3.13.
#   3. It answers the wrong question. GATE C asks "is a number produced HERE comparable to the
#      committed 92.3?" If it passes on Colab's own stack, comparability is demonstrated
#      empirically and the version difference is moot -- and this IS the stack we would use.
# The cause of a mismatch matters only for DIAGNOSIS, and §2's pixel fingerprint supplies that.
def version(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return None

import torch
actual = {m: version(m) for m in PINS}
print("\nstack here :", json.dumps(actual))
print("stack (Mac):", json.dumps(PINS))
print(f"  python {sys.version.split()[0]}   cuda {torch.version.cuda}   "
      f"gpu {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then re-run this cell.")

drift = {m: (actual[m], PINS[m]) for m in PINS if actual[m] != PINS[m]}
if drift:
    print(f"\n  NOTE: stack differs from the Mac's -- {drift}")
    print("  This is RECORDED, not fatal. §2 decides comparability empirically, and its pixel")
    print("  fingerprint separates a data-path difference from a GPU one if it fails.")

# ---- GATE A: the checkpoint is the one every committed number used ----------------------
# 🔴 progress.md §21.11 item 10 is WRONG TWICE, verified 2026-09-19:
#   (a) it names "getnexar/BADAS-Open" as the HF mirror. That is the GITHUB org. On Hugging
#       Face the model lives at "nexar-ai/BADAS-Open".
#   (b) it calls that mirror "ungated". It is GATED -- "You need to agree to share your contact
#       information to access this model". This repo's own .gitignore says "gated HF download"
#       and was right. HF returns 401 for BOTH a missing repo and a gated one, so the two
#       errors masked each other.
# Licence is Apache 2.0, free for research AND commercial use -- only ACCESS is the issue.
#
# Route 1 (preferred, no credentials): the checkpoint sitting in Drive beside the bundle.
# Route 2 (fallback): HF, needing accepted terms plus a read token in Colab Secrets.
# Either way GATE A's digest decides: the source is interchangeable, the bytes are not.
CKPT = os.path.join(REPO, "models", "badas", "weights", "badas_open.pth")
DRIVE_CKPT = os.path.join(DRIVE_DIR, "badas_open.pth")
os.makedirs(os.path.dirname(CKPT), exist_ok=True)

if not (os.path.exists(CKPT) and os.path.getsize(CKPT) == CKPT_BYTES):
    if os.path.exists(DRIVE_CKPT):
        print(f"copying checkpoint from Drive "
              f"({os.path.getsize(DRIVE_CKPT)/2**30:.2f} GiB) ...")
        shutil.copy2(DRIVE_CKPT, CKPT)
    else:
        err, files = None, []
        try:
            from huggingface_hub import hf_hub_download, list_repo_files
            tok = None
            try:
                from google.colab import userdata
                tok = userdata.get("HF_TOKEN")
            except Exception:
                pass
            files = list_repo_files("nexar-ai/BADAS-Open", token=tok)
            pth = [f for f in files if f.endswith(".pth")]
            if len(pth) != 1:
                raise RuntimeError(f"expected exactly one .pth, found {pth}")
            got = hf_hub_download(repo_id="nexar-ai/BADAS-Open", filename=pth[0], token=tok)
            shutil.copy2(got, CKPT)
        except Exception as e:
            err = e
        # Raised OUTSIDE the except block: a SystemExit chained off a live exception makes
        # IPython print ~200 lines of internal traceback and bury the actual instruction.
        if err is not None:
            raise SystemExit(
                f"Could not obtain the checkpoint.\n"
                f"  error   : {err}\n"
                f"  listing : {files or '<could not list -- gated, or not signed in>'}\n\n"
                "TWO WAYS TO FIX. Pick either.\n\n"
                "  A) DRIVE (no token, no terms, works every session)\n"
                f"     Upload the Mac's models/badas/weights/badas_open.pth (3.98 GB)\n"
                f"     into {DRIVE_DIR}/  then re-run this cell.\n"
                "     You already hold the exact verified bytes.\n\n"
                "  B) HUGGING FACE (fast, but gated)\n"
                "     Sign in at huggingface.co/nexar-ai/BADAS-Open and accept the\n"
                "     conditions. Make a READ token. In Colab click the key icon in the\n"
                "     left sidebar, add a secret named HF_TOKEN, enable notebook access.\n\n"
                "Do NOT substitute a different checkpoint -- GATE A would reject it anyway.")

got_sha, got_bytes = sha256(CKPT), os.path.getsize(CKPT)
if got_sha != CKPT_SHA or got_bytes != CKPT_BYTES:
    raise SystemExit(f"GATE A FAILED\n  sha256 {got_sha}\n  expect {CKPT_SHA}\n"
                     f"  bytes  {got_bytes} vs {CKPT_BYTES}\n"
                     "A different checkpoint means every number below is incomparable. HALT.")
print(f"GATE A  PASS -- checkpoint byte-identical ({got_bytes:,} bytes)")

NEXAR = next((p for p in NEXAR_CANDIDATES if os.path.isdir(os.path.join(p, "negative"))), None)
if NEXAR is None:
    hits = [os.path.dirname(p) for p in
            __import__("glob").glob("/content/drive/MyDrive/**/negative", recursive=True)]
    raise SystemExit(f"Nexar test-public not found. Tried {NEXAR_CANDIDATES}.\n"
                     f"Directories containing a 'negative/' folder: {hits[:10]}\n"
                     "Set NEXAR_CANDIDATES and re-run.")
n_neg = len([f for f in os.listdir(os.path.join(NEXAR, "negative")) if f.endswith(".mp4")])
print(f"nexar negatives: {n_neg} mp4 at {NEXAR}")
assert n_neg == 333, f"expected 333 Nexar negatives, found {n_neg} -- upload incomplete?"

PROVENANCE = {"stack": actual, "cuda": torch.version.cuda,
              "gpu": torch.cuda.get_device_name(0), "ckpt_sha256": got_sha}
print("\nNext: run §2. Nothing may be scored from comma2k19 until it passes.")


## 2 — GATE C: does a CUDA T4 reproduce the Mac's MPS scores? (decision D57)

**This runs before any comma2k19 frame is scored, and it cannot be skipped.**

100 Nexar negatives, chosen deterministically (`sorted(ids)`, `default_rng(0)`, 100 without
replacement — the same draw every time, on any machine), re-scored at stride 1 on the T4 and
compared per-clip against the committed MPS scores.

**Bar, declared before the numbers are read:** median |Δ| < 0.002 **and** at most 1 of 100
crossing the 0.9733 alert threshold.

`score_external.py --limit` takes the *first* N sorted clips, not a random sample, so the notebook
stages the chosen 100 into their own directory and then calls the committed script unchanged.

**First, a pixel fingerprint.** Before any GPU work, the cell decodes one clip through
`load_full_video_frames` and hashes the resulting frames. That quantity is pure cv2 decode +
resize + colour convert — **no GPU, no torch, no transformers touch it** — and the Mac's value is
embedded. So:

- fingerprint **matches** → the frames reaching the model are byte-identical on both machines, so
  any score difference is model/GPU numerics
- fingerprint **differs** → the decode path differs before the model even runs, and the GPU is
  exonerated

That single number turns a failure from "something is different" into a named cause, and it costs
one clip.

**On pass** it writes `EQUIVALENCE_PASSED.json`. **On failure it HALTs** and prints the matching
diagnosis. The contingency — a full 667-clip Colab sweep to re-derive the threshold, ~7 h —
**needs sign-off and is not started here.**


In [ ]:
# === §2 — GATE C: MPS vs T4 equivalence. Runs BEFORE any comma2k19 scoring (D57). ===
import glob
import numpy as np

MED_BAR, CROSS_BAR, THRESHOLD = 0.002, 1, 0.9733
SENTINEL = os.path.join(REPO, "EQUIVALENCE_PASSED.json")

# The Mac's decode+resample fingerprint for clip 01044 (first of the rng(0) sample), computed
# by load_full_video_frames(path, (224,224), 8.0) -> sha256 of the uint8 array. NO GPU, NO
# torch and NO transformers touch this quantity: it is pure cv2 decode + resize + colour
# convert. So if it MATCHES here, the data reaching the model is byte-identical on both
# machines and any score difference is model/GPU numerics. If it DIFFERS, the cause is the
# decode path (cv2 build / version) and the GPU is exonerated. That is the whole diagnosis,
# and it costs one clip.
MAC_PIXEL_FP = "bd6126c391b2301d170e40128d04338587a38697a7a494c63943d4182a23f24a"
MAC_FP_CLIP, MAC_FP_SHAPE = "01044", (81, 224, 224, 3)

ref = {}
with open(os.path.join(REPO, "runs/baselines/badas-open/scores.jsonl")) as f:
    for line in f:
        r = json.loads(line)
        if "score" in r:
            ref[r["id"]] = float(r["score"])
print(f"reference scores (MPS, committed): {len(ref)}")

neg_ids = sorted(os.path.splitext(f)[0]
                 for f in os.listdir(os.path.join(NEXAR, "negative")) if f.endswith(".mp4"))
assert len(neg_ids) == 333, len(neg_ids)
pick = sorted(np.random.default_rng(0).choice(np.array(neg_ids), size=100,
                                              replace=False).tolist())
print(f"deterministic sample: {len(pick)} clips, first 5 {pick[:5]}")

# ---- PIXEL FINGERPRINT FIRST. One clip, no GPU, seconds. Printed pass or fail. ----------
sys.path.insert(0, os.path.join(REPO, "vendor", "badas-open"))
from badas.utils.video import load_full_video_frames
_fr = load_full_video_frames(os.path.join(NEXAR, "negative", f"{MAC_FP_CLIP}.mp4"),
                             (224, 224), 8.0)
PIXEL_FP = hashlib.sha256(_fr.tobytes()).hexdigest()
PIXEL_MATCH = (PIXEL_FP == MAC_PIXEL_FP)
_verdict = ("IDENTICAL -- the decode path matches; only the model/GPU can differ" if PIXEL_MATCH
            else "DIFFERENT -- the DECODE path differs, before the model even runs")
print(f"\npixel fingerprint ({MAC_FP_CLIP}): shape {_fr.shape} vs Mac {MAC_FP_SHAPE}")
print(f"  here {PIXEL_FP}")
print(f"  mac  {MAC_PIXEL_FP}")
print(f"  -> {_verdict}")

STAGE = "/content/equiv_clips"
os.makedirs(STAGE, exist_ok=True)
for cid in pick:
    dst = os.path.join(STAGE, f"{cid}.mp4")
    if not os.path.exists(dst):
        shutil.copy2(os.path.join(NEXAR, "negative", f"{cid}.mp4"), dst)
print(f"staged {len(os.listdir(STAGE))} clips -> {STAGE}")

t0 = time.time()
r = subprocess.run([sys.executable, os.path.join(REPO, "scripts", "score_external.py"),
                    "--clips-dir", STAGE, "--out", "/content/equiv_out",
                    "--device", "cuda", "--stride", "1"],
                   capture_output=True, text=True, timeout=14400)
print(r.stdout[-2000:])
if r.returncode != 0:
    print("STDERR:\n", r.stderr[-2000:])
    raise SystemExit("equivalence scoring failed -- report back, do not improvise")
print(f"scored in {(time.time()-t0)/60:.1f} min")

got = {}
with open("/content/equiv_out/scores.jsonl") as f:
    for line in f:
        rec = json.loads(line)
        if "score" in rec:
            got[rec["id"]] = float(rec["score"])

common = sorted(set(got) & set(ref) & set(pick))
d = np.array([abs(got[c] - ref[c]) for c in common])
crossings = [c for c in common
             if (got[c] >= THRESHOLD) != (ref[c] >= THRESHOLD)]

print("\n" + "=" * 64)
print("GATE C -- MPS vs CUDA T4 EQUIVALENCE")
print("=" * 64)
print(f"  compared        {len(common)} clips")
print(f"  median |delta|  {np.median(d):.6f}   (bar < {MED_BAR})")
print(f"  max |delta|     {np.max(d):.6f}")
print(f"  crossings       {len(crossings)}     (bar <= {CROSS_BAR})   {crossings[:5]}")
print(f"  stack           {json.dumps(PROVENANCE['stack'])}")
print(f"  gpu             {PROVENANCE['gpu']}  cuda {PROVENANCE['cuda']}")

print(f"  pixel fp        {'MATCHES the Mac' if PIXEL_MATCH else 'DIFFERS from the Mac'}")

passed = float(np.median(d)) < MED_BAR and len(crossings) <= CROSS_BAR
if not passed:
    print("\n  GATE C FAILED.")
    print("  This hardware does not reproduce the Mac's scores within the declared bar, so the")
    print("  0.9733 threshold does not mean the same thing here, and comma2k19's FP/hour would")
    print("  NOT be comparable to the committed 92.3 -- which is the entire point of measuring.")
    print("\n  DIAGNOSIS, from the pixel fingerprint above:")
    if PIXEL_MATCH:
        print("    The decode path is byte-identical, so the frames reaching the model are the")
        print("    same on both machines. The difference is therefore in the MODEL/GPU numerics")
        print("    (CUDA vs MPS kernels, or the torch/transformers versions), not in the data.")
        print("    Next diagnostic: re-score the 10 worst clips on this same VM with")
        print("    --device cpu. CPU ~ CUDA and both != MPS -> library/arch. CPU ~ MPS and")
        print("    CUDA != both -> a genuine device difference.")
    else:
        print("    The decode path already DIFFERS before the model runs, so the GPU is")
        print("    exonerated -- this is cv2/decoder behaviour, not CUDA. Matching the Mac's")
        print("    opencv build is the lever, not anything about the device.")
    print("\n  CONTINGENCY (needs the user's sign-off, NOT started here): re-score all 667 Nexar")
    print("  clips on Colab (~7 h) and re-derive the threshold on this hardware.")
    raise SystemExit("GATE C FAILED -- HALT and report the numbers above.")

with open(SENTINEL, "w") as f:
    json.dump({"passed": True, "n": len(common), "median_abs_delta": float(np.median(d)),
               "max_abs_delta": float(np.max(d)), "crossings": len(crossings),
               "pixel_fp": PIXEL_FP, "pixel_fp_matches_mac": PIXEL_MATCH,
               **PROVENANCE}, f, indent=2)
print(f"\n  GATE C  PASS -- sentinel written to {SENTINEL}")
if drift:
    print("\n  Note on the stack difference recorded in §1: it is now MOOT for this purpose.")
    print("  Comparability was demonstrated empirically ON THIS STACK, which is the stack the")
    print("  comma2k19 numbers will be produced on. That is a stronger result than forcing the")
    print("  versions to match would have been.")
print("\nNext: §3. Scoring cells assert on that sentinel, so §2 cannot be skipped.")


## 2b — DIAGNOSTIC: *why* did GATE C fail, and is the pilot even affordable?

**Run this after a GATE C failure. It answers two questions at once and needs neither the
checkpoint nor a GPU**, so it survives a runtime restart and costs ~2 minutes.

**Question 1 — how do the two decode paths differ?** GATE C's fingerprint proved *that* they
differ; a hash cannot say *how*. This cell diffs the Mac's actual decoded array against
Colab's, element by element, and separates:

| what the diff shows | cause | how bad |
|---|---|---|
| max diff 1–2 grey levels, spread widely | `cv2.resize` / YUV→BGR rounding (NEON vs x86 SIMD) | **cosmetic** — same frames, ±1 noise |
| big diffs in whole frames, and Colab's frame *k* matches the Mac's frame *k±1* | `cap.set(CAP_PROP_POS_FRAMES)` seek accuracy differs across ffmpeg builds | **serious** — different footage |
| wholesale, no frame correspondence | different decoder | **serious** |

Needs `01044_frames.npz` in `MyDrive/crash_detection_colab/`, written on the Mac by
`scripts/export_pixel_reference.py`. Without it the cell still does Question 2.

**Question 2 — what does a comma2k19 segment actually cost?** GATE C measured **94.7 s/clip
over 100 clips at stride 1**, but decode and model time are collinear in that number and
cannot be separated from it. Timing decode *alone* separates them, and then the model's
per-window cost falls out by subtraction. A 60 s comma2k19 segment is **480 frames** against
Nexar's ~77, so this decides whether the §5 pilot can clear the 72 s bar — **before** anything
expensive is committed to. §21.12 item 7 predicted decode would dominate; this measures it.

In [ ]:
# === §2b — DIAGNOSTIC after a GATE C failure. No checkpoint, no GPU, ~2 min. ===
# Self-contained ON PURPOSE: it must run on a fresh runtime (e.g. straight after the §2c
# restart) without re-downloading a 3.98 GB checkpoint to answer a pure-cv2 question.
import hashlib, json, os, subprocess, sys, tempfile, time

REPO = "/content/repo"
DRIVE_DIR = "/content/drive/MyDrive/crash_detection_colab"
NEXAR_CANDIDATES = [
    "/content/drive/MyDrive/nexar/test-public",
    "/content/drive/MyDrive/test-public",
    "/content/drive/MyDrive/nexar_test_public",
]
MAC_PIXEL_FP = "bd6126c391b2301d170e40128d04338587a38697a7a494c63943d4182a23f24a"
MAC_FP_CLIP = "01044"

# GATE C's own measurements, 2026-09-19. Used to solve for the model's per-window cost.
GATEC_S_PER_CLIP, GATEC_MEAN_FRAMES = 94.7, 77.1
BAR_S_PER_SEGMENT = 72.0        # NEW_PLAN §13: 12 h / 600 one-minute segments
SEG_FRAMES = 60 * 8             # a 60 s comma2k19 segment at target_fps 8.0
WINDOW = 16                     # sliding window length; kept windows = n_frames - 16

if not os.path.ismount("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

import tarfile
if not os.path.exists(os.path.join(REPO, "vendor/badas-open/badas/utils/video.py")):
    bundle = os.path.join(DRIVE_DIR, "colab_bundle.tar.gz")
    if not os.path.exists(bundle):
        raise SystemExit(f"No bundle at {bundle}. Run §1 first, or re-upload it.")
    os.makedirs(REPO, exist_ok=True)
    with tarfile.open(bundle) as tar:
        tar.extractall(REPO)
    print(f"staged bundle -> {REPO}")

NEXAR = next((p for p in NEXAR_CANDIDATES if os.path.isdir(os.path.join(p, "negative"))), None)
if NEXAR is None:
    raise SystemExit(f"Nexar test-public not found. Tried {NEXAR_CANDIDATES}.")

import numpy as np
import cv2
print("=" * 70)
print("THE DECODE STACK HERE  (the thing GATE C's fingerprint implicated)")
print("=" * 70)
print(f"  cv2            {cv2.__version__}")
print(f"  numpy          {np.__version__}")
print("  mac            cv2 5.0.0  (opencv-python-headless 5.0.0.93), numpy 2.4.6")
_bi = cv2.getBuildInformation()
for key in ("FFMPEG", "avcodec", "avformat", "swscale"):
    for line in _bi.split("\n"):
        if key.lower() in line.lower():
            print(f"    {line.strip()[:96]}")
            break
print("  mac            avcodec 61.19.101  avformat 61.7.100  swscale 8.3.100")

# Import the VENDORED decoder. Never reimplement it -- a reimplementation would answer a
# question about itself rather than about the code that produced every committed score.
sys.path.insert(0, os.path.join(REPO, "vendor", "badas-open"))
try:
    from badas.utils.video import load_full_video_frames
except Exception as e:
    raise SystemExit(
        f"Could not import the vendored decoder: {e}\n"
        "If this appeared right after §2c, the opencv pin broke albumentations. Repair with:\n"
        "  !pip install -q 'albumentations' 'opencv-python-headless==5.0.0.93'\n"
        "then restart the runtime and re-run this cell. Do NOT reimplement the decoder.")

# ---------------------------------------------------------------------------------------
# QUESTION 1 -- how do the two decode paths differ?
# ---------------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("Q1  HOW THE DECODE PATHS DIFFER")
print("=" * 70)
here = load_full_video_frames(os.path.join(NEXAR, "negative", f"{MAC_FP_CLIP}.mp4"),
                              (224, 224), 8.0)
here_fp = hashlib.sha256(here.tobytes()).hexdigest()
print(f"  clip {MAC_FP_CLIP}   shape {here.shape}   dtype {here.dtype}")
print(f"  fingerprint here  {here_fp}")
print(f"  fingerprint mac   {MAC_PIXEL_FP}")
FP_MATCH = here_fp == MAC_PIXEL_FP
print(f"  -> {'IDENTICAL -- decode paths now agree' if FP_MATCH else 'STILL DIFFERENT'}")

ref_path = os.path.join(DRIVE_DIR, f"{MAC_FP_CLIP}_frames.npz")
VERDICT = None
if FP_MATCH:
    VERDICT = "identical"
    print("\n  Nothing to diff -- the arrays are byte-identical. GATE C can be re-run as-is.")
elif not os.path.exists(ref_path):
    print(f"\n  ⚠ No reference array at {ref_path}")
    print("  On the Mac: ~/envs/badas/bin/python scripts/export_pixel_reference.py")
    print("  then upload runs/pixel_ref/01044_frames.npz to that Drive folder.")
    print("  Q2 below does not need it.")
else:
    mac = np.load(ref_path)["frames"]
    print(f"\n  reference loaded: shape {mac.shape}")
    if mac.shape != here.shape:
        VERDICT = "shape"
        print(f"  🔴 SHAPES DIFFER {mac.shape} vs {here.shape} -- different frame COUNT, which "
              "would also corrupt the FP/hour denominator. Report this immediately.")
    else:
        a, b = mac.astype(np.int16), here.astype(np.int16)
        d = np.abs(a - b)
        pct_diff = 100.0 * (d > 0).mean()
        per_frame_max = d.reshape(len(d), -1).max(axis=1)
        print(f"  max |diff|            {d.max()} grey levels (of 255)")
        print(f"  mean |diff|           {d.mean():.4f}")
        print(f"  pixels differing      {pct_diff:.2f}%")
        print(f"  frames byte-identical {(per_frame_max == 0).sum()} / {len(d)}")
        print(f"  worst frames          {np.argsort(-per_frame_max)[:5].tolist()}"
              f" (max {per_frame_max.max()})")

        # THE decisive test: is Colab's frame k actually the Mac's frame k+-1?
        print("\n  frame-alignment test -- is this a SHIFT rather than rounding?")
        best = {}
        for off in (-2, -1, 0, 1, 2):
            tot, n = 0.0, 0
            for i in range(len(here)):
                j = i + off
                if 0 <= j < len(mac):
                    tot += np.abs(a[j] - b[i]).mean(); n += 1
            best[off] = tot / max(n, 1)
        for off in sorted(best):
            mark = "  <-- best" if best[off] == min(best.values()) else ""
            print(f"    mac[k{off:+d}] vs here[k]   mean |diff| {best[off]:8.4f}{mark}")
        best_off = min(best, key=best.get)

        if best_off != 0:
            VERDICT = "shift"
            print(f"\n  🔴 SERIOUS: frames align best at offset {best_off:+d}, NOT 0.")
            print("  Colab's seeking lands on DIFFERENT frames -- the two machines watched")
            print("  different footage. Matching median |delta| without fixing this would be")
            print("  papering over it.")
        elif d.max() <= 3:
            VERDICT = "rounding"
            print(f"\n  ✅ COSMETIC: aligned at offset 0, max difference only {d.max()} grey")
            print("  level(s). Same frames, SIMD/rounding noise. §2c's opencv pin is the")
            print("  targeted lever, and it is cheap to test.")
        else:
            VERDICT = "large-aligned"
            print(f"\n  🟡 Aligned at offset 0 but differences reach {d.max()} grey levels --")
            print("  larger than rounding. Suspect the decoder or colour conversion, not the")
            print("  seek. Report the numbers; do not guess a fix.")

# ---------------------------------------------------------------------------------------
# QUESTION 2 -- decode cost, and therefore whether the pilot is affordable
# ---------------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("Q2  DECODE COST -- CAN A 60 s SEGMENT CLEAR THE 72 s BAR?")
print("=" * 70)
neg = sorted(f for f in os.listdir(os.path.join(NEXAR, "negative")) if f.endswith(".mp4"))
sizes = [(f, os.path.getsize(os.path.join(NEXAR, "negative", f))) for f in neg]
sizes.sort(key=lambda x: x[1])
probe = [sizes[0][0], sizes[len(sizes) // 2][0], sizes[-1][0]]   # small, median, large

def timed_decode(path):
    t0 = time.perf_counter()
    fr = load_full_video_frames(path, (224, 224), 8.0)
    return len(fr), time.perf_counter() - t0

print("  decode ONLY (no model), Nexar clips:")
rows = []
for f in probe:
    n, secs = timed_decode(os.path.join(NEXAR, "negative", f))
    rows.append((f, n, secs))
    print(f"    {f:<12} {n:4d} frames  {secs:7.2f} s   {secs / n:.4f} s/frame")

# A ~480-frame file, built LOSSLESSLY (-c copy, no re-encode) so decode behaviour is
# comparable. This is NOT the forbidden 8 fps transcode -- no pixel is re-encoded, and the
# file is a throwaway timing probe that never reaches the scorer.
long_src = os.path.join(NEXAR, "negative", sizes[-1][0])
reps = max(2, int(round(SEG_FRAMES / max(rows[-1][1], 1))))
TMP = "/content" if os.path.isdir("/content") else tempfile.mkdtemp()
lst, cat = os.path.join(TMP, "_cat.txt"), os.path.join(TMP, "_long_probe.mp4")
with open(lst, "w") as fh:
    fh.write("".join(f"file '{long_src}'\n" for _ in range(reps)))
try:
    cp = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0",
                         "-i", lst, "-c", "copy", cat], capture_output=True, text=True)
    cp_err = cp.stderr if cp.returncode else ""
except FileNotFoundError:
    cp, cp_err = None, "ffmpeg not on PATH (it IS present on Colab; this is a local run)"
long_row = None
if cp is not None and cp.returncode == 0 and os.path.exists(cat):
    n, secs = timed_decode(cat)
    long_row = (n, secs)
    print(f"\n  a LONG file ({reps}x lossless concat, ~a comma2k19 segment):")
    print(f"    {n:4d} frames  {secs:7.2f} s   {secs / n:.4f} s/frame")
    short_pf = rows[-1][2] / rows[-1][1]
    print(f"    per-frame cost vs the {rows[-1][1]}-frame original: "
          f"{secs / n / short_pf:.2f}x  "
          f"({'grows with file length -- seeks get more expensive' if secs / n > short_pf * 1.25 else 'stable with file length'})")
else:
    print(f"\n  ⚠ could not build the long probe: {cp_err[-300:]}")
    print("    -> the 480-frame projection below falls back to EXTRAPOLATING the short\n          clips, which the growth trend above shows is optimistic. Say so if you quote it.")

# Per-frame decode cost is NOT constant -- it grows with clip length, because seeking
# further into a file means re-decoding from a more distant keyframe. Show the trend, then
# use the probe CLOSEST IN LENGTH to GATE C's mean clip rather than a median of rates.
pfs = [(n, s / n) for _, n, s in rows]
print("  per-frame decode cost vs clip length (does seeking get more expensive?):")
for n, r in pfs:
    print(f"    {n:4d} frames -> {r:.4f} s/frame")
growth = pfs[-1][1] / pfs[0][1]
print(f"    {growth:.2f}x from shortest to longest -> "
      f"{'GROWS with length; short-clip rates UNDERSTATE a 60 s segment' if growth > 1.25 else 'roughly flat'}")
pf = min(pfs, key=lambda t: abs(t[0] - GATEC_MEAN_FRAMES))[1]
print(f"  using {pf:.4f} s/frame (probe nearest GATE C's {GATEC_MEAN_FRAMES:.0f}-frame mean)")
gatec_decode = pf * GATEC_MEAN_FRAMES
gatec_windows = GATEC_MEAN_FRAMES - WINDOW
per_window = (GATEC_S_PER_CLIP - gatec_decode) / gatec_windows
print("\n  separating GATE C's 94.7 s/clip into its two parts:")
print(f"    decode  {gatec_decode:6.1f} s  ({100 * gatec_decode / GATEC_S_PER_CLIP:.0f}% of the total)")
print(f"    model   {GATEC_S_PER_CLIP - gatec_decode:6.1f} s"
      f"  over {gatec_windows:.0f} windows = {per_window:.3f} s/window")
if per_window <= 0:
    print("    🔴 NEGATIVE model cost -- decode alone exceeds GATE C's total. Either this VM")
    print("    is slower than the one GATE C ran on, or the timing is not comparable. Say so;")
    print("    do not extrapolate from it.")

seg_decode = (long_row[1] / long_row[0] * SEG_FRAMES) if long_row else pf * SEG_FRAMES
seg_windows = (SEG_FRAMES - WINDOW) / 8.0        # stride 8 == 1 Hz
seg_total = seg_decode + seg_windows * max(per_window, 0.0)
print(f"\n  PROJECTED 60 s comma2k19 segment at stride 8 ({SEG_FRAMES} frames, "
      f"{seg_windows:.0f} scored windows):")
print(f"    decode {seg_decode:6.0f} s  +  model {seg_windows * max(per_window, 0.0):5.0f} s"
      f"  =  {seg_total:6.0f} s/segment")
print(f"    bar    {BAR_S_PER_SEGMENT:6.0f} s/segment   ->  {seg_total / BAR_S_PER_SEGMENT:.1f}x the bar")
print(f"    600 segments (10 h of footage) = {seg_total * 600 / 3600:.0f} h"
      f"   (NEW_PLAN §13 kills comma2k19 above 12 h)")
print(f"\n  PILOT VERDICT: {'AFFORDABLE' if seg_total <= BAR_S_PER_SEGMENT else 'OVER THE BAR'}"
      f" -- {'proceed to §5 once GATE C passes' if seg_total <= BAR_S_PER_SEGMENT else 'report to the user and re-plan; do NOT transcode to 8 fps (lossy, breaks GATE C comparability)'}")

print("\n" + "=" * 70)
print(f"SUMMARY   decode-diff verdict: {VERDICT or 'not run (no reference array)'}"
      f"   |   pilot: {seg_total / BAR_S_PER_SEGMENT:.1f}x bar")
print("Report BOTH lines to the user. Nothing expensive starts on this cell's word.")
print("=" * 70)


## 2c — the approved targeted OpenCV test

**User-approved 2026-09-19.** GATE C's fingerprint identified the **decode path** as the
cause, so this installs the Mac's exact OpenCV and re-checks the fingerprint.

> **This does not change decision D61, and D61 stays on the record.** D61 rejected a
> *speculative, untargeted* pin of `torch`/`transformers`/`numpy`, made before any evidence,
> on modules Colab preloads (so `pip` could not change the running kernel) at a ~2.5 GB cost.
> This is different on every count: the fingerprint has **identified** `cv2` as the actual
> cause, `cv2` is **not** preloaded, it is ~90 MB, and it is verified by the **fingerprint in
> seconds** rather than by a 2.6 h scoring run. Targeted diagnostic, not a stack pin.

**After running this, restart the runtime, then re-run §2b** — which is self-contained and
needs no checkpoint. Only if §2b reports the fingerprints **IDENTICAL** is it worth spending
2.6 h on GATE C again. **GATE C's bar is untouched.**

In [ ]:
# === §2c — install the Mac's exact OpenCV, then RESTART and re-run §2b. ===
# Mac: opencv-python-headless 5.0.0.93 -> cv2 5.0.0, avcodec 61.19.101, swscale 8.3.100.
# Colab ships its own opencv build; remove it first or two cv2 modules fight over the import.
!pip uninstall -y -q opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless
!pip install -q "opencv-python-headless==5.0.0.93"
# albumentations imports cv2 at module scope and may have pinned the version we just removed.
!pip install -q --no-deps albumentations || true
import subprocess, sys
print(subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True).stdout[-1500:])
print("""
==============================================================
NOW: Runtime > Restart session, then re-run §2b (ONLY §2b).
§2b is self-contained -- no checkpoint, no GPU, ~2 minutes.

  fingerprints IDENTICAL -> the decode path is fixed. Re-run §1 then §2 (GATE C, 2.6 h)
                            with its bar UNCHANGED.
  still DIFFERENT        -> stop. Report §2b's numbers. Do not iterate on library versions
                            blind; the remaining options need the user's sign-off.
==============================================================""")


## 3 — Acquire comma2k19

**Verified 2026-09-18 against primary sources**, not taken from `progress.md`:

| | |
|---|---|
| Licence | **MIT** (repo `LICENSE` + HF metadata) — commercial use permitted |
| Source | HF dataset `commaai/comma2k19`, `raw_data/Chunk_1.zip` … `Chunk_10.zip` |
| Size | 8.73–9.9 GB per chunk, **94.6 GB** total |
| Content | 10 chunks × ~200 one-minute segments = **~3.33 h each**; 3 chunks ≈ 10 h |
| Segment | `preview.png`, `raw_log.bz2`, `video.hevc`, `processed_log/`, **`global_pos/`** |
| `frame_times` | in `global_pos/`, timestamps in **boot time (seconds)** |
| `processed_log/` | IMU (accel, gyro, magnetic) + CAN (car_speed, steering_angle, wheel_speeds, radar) + GNSS → **R9's kill condition will not fire** |

🔴 **`progress.md` was wrong, and this is the correction.** §13 and §17-S14 both say
`global_pose/frame_times`. The real directory is **`global_pos/`** — §21.11 had it right. That
contradiction sat exactly at the frame-rate seam. The cell still *discovers* the layout and HALTs
rather than trusting even this table.

Archives are **ZIP**, so members list from the central directory without extracting — a wrong
archive fails in seconds rather than after a 10 GB unpack.

Disk: one chunk at a time — **download → extract → remux → score → delete before the next**.
Peak ≈ 22 GB, checked before the download starts.


In [ ]:
# === §3 — acquire chunk 1, and DISCOVER the layout rather than assume it. ===
assert os.path.exists(SENTINEL), "GATE C has not passed -- run §2 first (D57)."

# VERIFIED 2026-09-18 against primary sources (github.com/commaai/comma2k19 README + LICENSE,
# and the HF dataset's own file tree): licence MIT; raw_data/Chunk_1.zip .. Chunk_10.zip,
# 8.73-9.9 GB each, 94.6 GB total; 10 chunks of ~200 one-minute segments = ~3.33 h each.
# ZIP, not tar -- so members can be listed from the central directory without extracting.
C2K_REPO, C2K_CHUNKS = "commaai/comma2k19", [f"raw_data/Chunk_{i}.zip" for i in range(1, 11)]
CHUNK = C2K_CHUNKS[0]
WORK, RAW = "/content/c2k_work", "/content/c2k_raw"

free = shutil.disk_usage("/content").free
need = 22e9          # ~10 GB archive + ~10 GB extracted, plus headroom
print(f"disk free {free/2**30:.0f} GiB   one download->extract cycle peaks at ~{need/2**30:.0f} GiB")
if free < need:
    raise SystemExit(f"only {free/2**30:.0f} GiB free, the cycle needs ~{need/2**30:.0f} GiB. "
                     "Report back -- do not try to stream around this.")

os.makedirs(RAW, exist_ok=True); os.makedirs(WORK, exist_ok=True)
from huggingface_hub import hf_hub_download
t0 = time.time()
arch = hf_hub_download(repo_id=C2K_REPO, filename=CHUNK, repo_type="dataset", local_dir=RAW)
print(f"downloaded {os.path.getsize(arch)/2**30:.1f} GiB in {(time.time()-t0)/60:.1f} min")

# ---- list members BEFORE extracting. Cheap on a zip, and it fails fast on a wrong archive.
import zipfile
with zipfile.ZipFile(arch) as z:
    names = z.namelist()
print(f"{len(names):,} members. first 5: {names[:5]}")
if not any(n.endswith("video.hevc") for n in names):
    raise SystemExit(f"no video.hevc among {len(names)} members -- this is not comma2k19.\n"
                     f"first 20: {names[:20]}\nReport back, do not improvise.")

shutil.unpack_archive(arch, WORK)
os.remove(arch)                                   # reclaim ~10 GB immediately

# ---- DISCOVER the layout. Never assume a spelling. --------------------------------------
hevc = sorted(glob.glob(os.path.join(WORK, "**", "video.hevc"), recursive=True))
if not hevc:
    tree = sorted(glob.glob(os.path.join(WORK, "**"), recursive=True))[:40]
    raise SystemExit("No video.hevc anywhere in the archive. First 40 paths:\n  " +
                     "\n  ".join(tree) + "\nReport back -- this is not the expected dataset.")

seg0 = os.path.dirname(hevc[0])
siblings = sorted(os.listdir(seg0))
print(f"segments found: {len(hevc)}")
print(f"one segment ({seg0}) holds: {siblings}")

# 'global_pos' is the VERIFIED spelling (comma2k19's own README). progress.md §21.11 has it
# right; §13 and §17-S14 both say 'global_pose' and are WRONG. Both are tried anyway -- the
# check costs nothing and the documentation has already been wrong once here.
POSE_DIR = next((d for d in ("global_pos", "global_pose") if d in siblings), None)
if POSE_DIR is None:
    raise SystemExit(f"Neither 'global_pos' nor 'global_pose' present. Segment holds: {siblings}\n"
                     "Report back -- do not pick a different file to derive the frame rate from.")
times_files = sorted(os.listdir(os.path.join(seg0, POSE_DIR)))
print(f"{POSE_DIR}/ holds: {times_files}")
TIMES = next((f for f in times_files if "frame_times" in f), None)
if TIMES is None:
    raise SystemExit(f"No frame_times in {POSE_DIR}/. Holds: {times_files}\n"
                     "The frame rate CANNOT be derived without it. Report back.")
assert "processed_log" in siblings, f"no processed_log/ -- R9's IMU/CAN precondition fails: {siblings}"
print(f"\nlayout CONFIRMED by discovery: {POSE_DIR}/{TIMES}, processed_log/ present")
print("Next: §3b (partition), then §4 (frame-rate gate).")


## 3b — IMU/CAN and R9's partition, assigned **before any score is seen** (decision D58)

`NEW_PLAN.md` R9 needs three disjoint thirds — **mine** (find hard false positives), **fit**
(train the rejection classifier), **report** (the final number) — and warns that collapsing any two
inflates the headline.

The partition is assigned **by route id** (so segments of one drive cannot straddle thirds) with
`rng(0)`, and written into `manifest.json` **now**, before anything is scored.
**A partition chosen after seeing scores is not a partition.**


In [ ]:
# === §3b — R9 three-way partition by ROUTE, seed 0, BEFORE any score exists (D58). ===
segments = [os.path.dirname(p) for p in hevc]
# route id = the directory above the segment number, e.g. .../<route>/<segment>/video.hevc
routes = sorted({os.path.basename(os.path.dirname(s)) for s in segments})
print(f"{len(segments)} segments across {len(routes)} routes")

rng = np.random.default_rng(0)
shuffled = list(rng.permutation(np.array(routes)))
third = len(shuffled) // 3
PARTITION = {}
for i, route in enumerate(shuffled):
    PARTITION[str(route)] = "mine" if i < third else ("fit" if i < 2 * third else "report")

counts = {k: sum(1 for v in PARTITION.values() if v == k) for k in ("mine", "fit", "report")}
print(f"routes per partition: {counts}")
assert len(set(PARTITION.values())) == 3, "all three partitions must be populated"

# IMU/CAN presence, verified against the actual bytes rather than the documentation.
log0 = os.path.join(segments[0], "processed_log")
signals = sorted(os.listdir(log0)) if os.path.isdir(log0) else []
print(f"processed_log/ holds: {signals}")
if not signals:
    print("  WARNING: processed_log/ is empty -- R9's kill condition ('release ships no IMU/CAN')")
    print("  WOULD fire. Record this; it does not block the FP/hour measurement.")

with open("/content/partition.json", "w") as f:
    json.dump({"seed": 0, "by": "route", "counts": counts, "routes": PARTITION,
               "imu_signals": signals}, f, indent=2)
print("\npartition written BEFORE any scoring. Next: §4.")


## 4 — GATE D: derive the frame rate, never assume it

`video.hevc` is a **raw HEVC elementary stream** — no container, no timestamps. ffmpeg must be
*told* a rate, and if we tell it the wrong one the model watches the road at the wrong speed.

**Why this gate is load-bearing:** `eval/fp_rate.py` computes `duration_s = len(scores)/target_fps`.
A wrong input rate corrupts **both** the false-positive count *and* the hours denominator — the
rate would be wrong twice over, and nothing would raise an error.

So: derive `fps = (n-1)/(t[-1]-t[0])` from the pose file, remux `-c copy` (lossless), then
**HALT** unless the decoded frame count matches `len(frame_times)` and the decoded duration agrees
to ±0.05 s. Proved on **one** segment first, with timing.


In [ ]:
# === §4 — GATE D: frame rate derived per segment, then proved on ONE with timing. ===
import numpy as np

def frame_times(seg):
    p = os.path.join(seg, POSE_DIR, TIMES)
    t = np.load(p) if p.endswith(".npy") else np.loadtxt(p)
    return np.asarray(t, float).ravel()

def derive_fps(seg):
    t = frame_times(seg)
    if t.size < 2:
        raise RuntimeError(f"{seg}: {t.size} frame times -- cannot derive a rate")
    return (t.size - 1) / (t[-1] - t[0]), t

def remux(seg, out):
    """Lossless container wrap at the DERIVED rate. -c copy: no re-encode, no quality loss."""
    fps, t = derive_fps(seg)
    r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "hevc",
                        "-r", f"{fps:.6f}", "-i", os.path.join(seg, "video.hevc"),
                        "-c", "copy", out], capture_output=True, text=True, timeout=1800)
    if r.returncode != 0:
        raise RuntimeError(f"ffmpeg failed on {seg}:\n{r.stderr[-1500:]}")
    return fps, t

# ---- PROVE IT ON ONE ----
import cv2
VID = "/content/c2k_mp4"; os.makedirs(VID, exist_ok=True)
s0 = segments[0]
out0 = os.path.join(VID, "probe.mp4")
t0 = time.time()
fps0, t_arr = remux(s0, out0)
dt = time.time() - t0

# 🔴 READ BACK THROUGH cv2, NOT ffprobe, AND NOT THE RATE WE ASKED FOR.
# vendor/badas-open/badas/utils/video.py::load_full_video_frames does exactly this:
#     video_duration     = CAP_PROP_FRAME_COUNT / CAP_PROP_FPS      <- the CONTAINER's numbers
#     target_frame_count = round(video_duration * 8.0)              <- becomes len(scores)
# and eval/fp_rate.py then computes duration_s = len(scores)/8.0. So whatever cv2 reads out of
# the container IS the FP/hour denominator. Checking the rate we *passed* to ffmpeg would prove
# nothing -- if ffmpeg wrote something else, cv2 would believe ffmpeg and we would never know.
cap = cv2.VideoCapture(out0)
ok, _ = cap.read()
n_cv2 = cap.get(cv2.CAP_PROP_FRAME_COUNT)
fps_cv2 = cap.get(cv2.CAP_PROP_FPS)
cap.release()
dur_cv2 = n_cv2 / max(fps_cv2, 1e-9)
dur_times = (t_arr[-1] - t_arr[0]) * t_arr.size / max(t_arr.size - 1, 1)

print("=" * 64)
print("GATE D -- FRAME RATE, DERIVED AND READ BACK THROUGH cv2")
print("=" * 64)
print(f"  derived fps (pose)   {fps0:.4f}   from {t_arr.size} frame times")
print(f"  cv2 reports fps      {fps_cv2:.4f}   <- THIS is what the model will believe")
print(f"  cv2 frame count      {n_cv2:.0f}   vs frame_times {t_arr.size}")
print(f"  cv2 duration         {dur_cv2:.3f}s   vs frame_times span {dur_times:.3f}s")
print(f"  |difference|         {abs(dur_cv2 - dur_times):.4f}s   (bar 0.05s)")
print(f"  implied len(scores)  {round(dur_cv2 * 8.0)}   -> fp_rate hours = "
      f"{round(dur_cv2 * 8.0) / 8.0 / 3600:.6f}")
print(f"  remux time           {dt:.1f}s   ({os.path.getsize(out0)/2**20:.1f} MiB)")

assert ok, "remuxed mp4 will not decode -- stop"
assert 5.0 <= fps0 <= 60.0, f"derived fps {fps0:.4f} is implausible for dashcam video"
d = np.diff(t_arr)
assert (d > 0).all(), "frame_times not monotonic -- a constant-rate remux would misplace frames"
if abs(fps_cv2 - fps0) > 0.01:
    raise SystemExit(f"GATE D FAILED: we asked ffmpeg for {fps0:.4f} fps but cv2 reads "
                     f"{fps_cv2:.4f}. The container disagrees with the derivation and cv2 is "
                     "what the model trusts. HALT.")
if abs(n_cv2 - t_arr.size) > 1:
    raise SystemExit(f"GATE D FAILED: cv2 counts {n_cv2:.0f} frames vs {t_arr.size} frame times. "
                     "The rate or the stream is wrong; every score would be silently invalid.")
if abs(dur_cv2 - dur_times) > 0.05:
    raise SystemExit(f"GATE D FAILED: duration disagrees by {abs(dur_cv2-dur_times):.3f}s "
                     "(bar 0.05s). fp_rate derives its HOURS denominator from this, so both the "
                     "FP count and the rate would be wrong. HALT.")
print("\n  GATE D  PASS -- the time base survives hevc -> mp4 -> cv2 intact")
os.remove(out0)
print("Next: §5, the 20-segment pilot.")


## 5 — Pilot: 20 segments, stride 8

`--stride 8` at `target_fps 8.0` is **exactly a 1 Hz alert cadence** (NEW_PLAN §7.3).

**Stop condition, stated concretely.** `NEW_PLAN.md` §13: throughput implying > 12 h for 10 h of
footage means stop and re-plan. 10 h = **600 one-minute segments**, so the bar is
**72 s/segment**. The cell prints the measured rate beside it.

🔴 **This may well be the cell that fails.** `load_full_video_frames` seeks with
`cap.set(CAP_PROP_POS_FRAMES, i)` for *every* sampled frame — ~480 random seeks into a
1200-frame HEVC stream per segment. Decode, not the GPU, is the likely bottleneck, and the
estimate straddles the 72 s bar. Only the measurement settles it.

⚠️ **The obvious fix is not available silently.** Transcoding to 8 fps at remux time would
remove the seeks, but it is **lossy** and the Nexar baseline was scored from natively-encoded
video — so it would break the comparability that gate C exists to protect. If the bar is missed,
that option needs its own mini-equivalence test and your approval. Do not take it here.

**Then stop and report the number before running §6.**


In [ ]:
# === §5 — pilot 20 segments at 1 Hz. Measure, never assert. ===
assert os.path.exists(SENTINEL), "GATE C has not passed -- run §2 first (D57)."

BAR_S_PER_SEGMENT = 72.0          # 12 h / 600 one-minute segments = the NEW_PLAN §13 bar
PILOT_N = 20
OUT = "/content/c2k_out"

CV2_DUR = {}      # sid -> the duration cv2 reports, which IS fp_rate's hours denominator

def prepare(seg):
    """Remux one segment and run GATE D on it. Returns sid, or raises."""
    sid = f"{os.path.basename(os.path.dirname(seg))}_{os.path.basename(seg)}"
    dst = os.path.join(VID, f"{sid}.mp4")
    if not (os.path.exists(dst) and os.path.getsize(dst) > 0):
        fps, t = remux(seg, dst)
    else:
        fps, t = derive_fps(seg)
    cap = cv2.VideoCapture(dst)
    nd, fd = cap.get(cv2.CAP_PROP_FRAME_COUNT), cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    assert abs(nd - t.size) <= 1, f"{sid}: cv2 counts {nd:.0f} frames vs {t.size} frame times"
    assert abs(fd - fps) <= 0.01, f"{sid}: cv2 reads {fd:.4f} fps, derived {fps:.4f}"
    CV2_DUR[sid] = nd / max(fd, 1e-9)
    return sid

pilot = segments[:PILOT_N]
built, failed = [], []
t0 = time.time()
for n, seg in enumerate(pilot, 1):
    try:
        built.append(prepare(seg))
    except Exception as e:
        sid = f"{os.path.basename(os.path.dirname(seg))}_{os.path.basename(seg)}"
        failed.append((sid, str(e)[:200]))
        print(f"  {n}/{len(pilot)}  {sid}  FAILED: {str(e)[:120]}", flush=True)
        continue
    if n % 10 == 0 or n == len(pilot):
        el = time.time() - t0
        print(f"  {n}/{len(pilot)}  {el/60:.1f} min elapsed, "
              f"eta {el/n*(len(pilot)-n)/60:.1f} min", flush=True)
print(f"\nremuxed {len(built)}/{len(pilot)}  failed {len(failed)}")

t1 = time.time()
r = subprocess.run([sys.executable, os.path.join(REPO, "scripts", "score_external.py"),
                    "--clips-dir", VID, "--out", OUT, "--device", "cuda", "--stride", "8"],
                   capture_output=True, text=True, timeout=14400)
print(r.stdout[-1500:])
if r.returncode != 0:
    print("STDERR:\n", r.stderr[-1500:])
    raise SystemExit("pilot scoring failed -- report back")

elapsed = time.time() - t1
per_seg = elapsed / max(len(built), 1)

# ---- 🔴 THE END-TO-END RATE CHECK. This is the one that would have caught a wrong rate. ----
# fp_rate computes hours as len(scores)/8.0. Compare that against what cv2 says the footage
# actually is. One 8 Hz quantum is 0.125 s, so 0.13 s is the tightest honest tolerance.
bad_rate = []
for sid in built:
    p = os.path.join(OUT, "frames", f"{sid}.npz")
    if not os.path.exists(p):
        continue
    with np.load(p) as f:
        implied = len(f["scores"]) / float(f["target_fps"])
    if abs(implied - CV2_DUR[sid]) > 0.13:
        bad_rate.append((sid, implied, CV2_DUR[sid]))
if bad_rate:
    for sid, a, b in bad_rate[:5]:
        print(f"  {sid}: trace implies {a:.2f}s of footage, container says {b:.2f}s")
    raise SystemExit(f"{len(bad_rate)}/{len(built)} segments disagree on duration. "
                     "fp_rate's hours denominator comes from len(scores)/8.0, so this run "
                     "would report the wrong FP/hour twice over. HALT and report back.")
print(f"end-to-end rate check: {len(built)}/{len(built)} traces agree with the container "
      f"to within 0.13s")

print("\n" + "=" * 64)
print("PILOT THROUGHPUT")
print("=" * 64)
print(f"  scored             {len(built)} segments in {elapsed/60:.1f} min")
print(f"  MEASURED           {per_seg:.1f} s/segment")
print(f"  stop-condition bar {BAR_S_PER_SEGMENT:.0f} s/segment  (= 12 h for 600 segments)")
print(f"  projected for 10 h {per_seg*600/3600:.1f} h")
if per_seg > BAR_S_PER_SEGMENT:
    print("\n  STOP CONDITION HIT (NEW_PLAN.md 13). Report this number and re-plan.")
    print("  Do NOT start 6.")
else:
    print(f"\n  under the bar by {BAR_S_PER_SEGMENT - per_seg:.0f} s/segment")
print("\nReport the measured rate to the user and WAIT for approval before running 6.")


## 6 — Full run: 3 chunks ≈ 10 h of footage

Chunk-by-chunk, each deleted before the next. Resumable — `score_external.py` skips ids already in
`scores.jsonl`, so a Colab disconnect costs only the segments in flight.

**Run this only after the pilot number has been reported and approved.**


In [ ]:
# === §6 — full run. Only after the pilot was reported AND approved. ===
assert os.path.exists(SENTINEL), "GATE C has not passed -- run §2 first (D57)."
APPROVED_AFTER_PILOT = False      # set True only once the user has seen §5's number

if not APPROVED_AFTER_PILOT:
    raise SystemExit("Set APPROVED_AFTER_PILOT = True only after reporting §5's measured "
                     "s/segment to the user and getting approval. NEW_PLAN.md 13 requires it.")

t0 = time.time()
todo = [s for s in segments if s not in {os.path.dirname(p) for p in hevc[:PILOT_N]}]
for n, seg in enumerate(todo, 1):
    try:
        built.append(prepare(seg))          # same GATE D checks as the pilot, per segment
    except Exception as e:
        sid = f"{os.path.basename(os.path.dirname(seg))}_{os.path.basename(seg)}"
        failed.append((sid, str(e)[:200]))
        print(f"  {n}/{len(todo)}  {sid}  FAILED: {str(e)[:120]}", flush=True)
        continue
    if n % 10 == 0 or n == len(todo):
        el = time.time() - t0
        print(f"  {n}/{len(todo)}  {el/60:.1f} min elapsed, "
              f"eta {el/n*(len(todo)-n)/60:.1f} min", flush=True)

r = subprocess.run([sys.executable, os.path.join(REPO, "scripts", "score_external.py"),
                    "--clips-dir", VID, "--out", OUT, "--device", "cuda", "--stride", "8"],
                   capture_output=True, text=True, timeout=72000)
print(r.stdout[-2500:])
if r.returncode != 0:
    print("STDERR:\n", r.stderr[-2000:])
    raise SystemExit("full run failed -- re-run this cell, it resumes from scores.jsonl")
print(f"\nfull run done in {(time.time()-t0)/3600:.1f} h")


## 7 — Manifest, and bring the traces home

**Video does not come back.** Traces are tiny — 220 DADA traces were 880 KB — and every number is
computed on the Mac by `eval/fp_rate.py`, the committed code that already reproduces Nexar's 92.3.

The manifest uses the nested shape `scripts/verify_manifest.py` reads unchanged.


In [ ]:
# === §7 — manifest + copy traces to Drive. Video stays here. ===
DEST = "/content/drive/MyDrive/comma2k19_fp"
os.makedirs(DEST, exist_ok=True)

traces = sorted(glob.glob(os.path.join(OUT, "frames", "*.npz")))
man = {}
for p in traces:
    man[os.path.basename(p)] = {"sha256": sha256(p), "bytes": os.path.getsize(p)}
with open(os.path.join(OUT, "manifest.json"), "w") as f:
    json.dump({"fps": 8.0, "n": len(man), "failed": failed, "clips": man,
               "partition": json.load(open("/content/partition.json")),
               "provenance": PROVENANCE,
               "equivalence": json.load(open(SENTINEL))}, f, indent=2)

for name in ("scores.jsonl", "manifest.json"):
    shutil.copy2(os.path.join(OUT, name), os.path.join(DEST, name))
os.makedirs(os.path.join(DEST, "frames"), exist_ok=True)
for p in traces:
    d = os.path.join(DEST, "frames", os.path.basename(p))
    if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(p):
        shutil.copy2(p, d)

tot = sum(os.path.getsize(os.path.join(DEST, "frames", f))
          for f in os.listdir(os.path.join(DEST, "frames")))
print(f"copied {len(traces)} traces ({tot/2**20:.1f} MiB) + scores.jsonl + manifest.json -> {DEST}")
print(f"failures recorded in the manifest: {len(failed)}")
print("\nNext, ON THE MAC:")
print("  1. download MyDrive/comma2k19_fp -> runs/comma2k19/")
print("  2. ~/envs/badas/bin/python scripts/verify_manifest.py --dir runs/comma2k19/frames")
print("  3. ~/envs/badas/bin/python -m eval.fp_rate \\")
print("         --frames-dir runs/comma2k19/frames --label comma2k19")
print("     B is the headline, A beside it, both with denominators.")
print("     Highway-only is a FLOOR, never a general rate (NEW_PLAN.md 8.2).")
print("     NEVER pool with ZOD.")


## RESUME — after a Colab disconnect

Colab drops runtimes. This re-establishes state without redoing finished work.


In [ ]:
# === RESUME: remount, re-stage, prove readability, report what is left. ===
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

print("1. sentinel :", "PRESENT" if os.path.exists(SENTINEL) else "GONE -- re-run 1 and 2")
done = set()
if os.path.exists(os.path.join(OUT, "scores.jsonl")):
    with open(os.path.join(OUT, "scores.jsonl")) as f:
        for line in f:
            try:
                done.add(json.loads(line)["id"])
            except Exception:
                continue
print(f"2. scored   : {len(done)} segments already in scores.jsonl (they will be skipped)")
print(f"3. mp4s     : {len(glob.glob(os.path.join(VID, '*.mp4')))} remuxed on local disk")

# A file can exist and still be truncated -- spot-check one finished artifact.
if done:
    p = sorted(glob.glob(os.path.join(OUT, "frames", "*.npz")))[0]
    with np.load(p) as f:
        assert f["scores"].size > 0 and float(f["target_fps"]) == 8.0, "a finished trace is bad"
    print(f"4. spotcheck: {os.path.basename(p)} decodes, fps 8.0 -- OK")
print("\n-> re-run the cell you were in; both remux and scoring skip completed work.")
